# Use Case 1 - Revenue & Growth Analytics

```
Summary : Exploration, validation and business-question answers for Use Case 1 (Revenue & Growth Analytics). Runs sanity SQL against bronze / silver / gold tables produced by the DLT pipelines in `transformations/silver/revenue/revenue_*.py` and `transformations/gold/revenue_kpis.py`.
Data    : raw_orders, raw_order_items, raw_order_payments, raw_products, raw_product_names_translation, raw_customers
```

## Hypothesis
E-commerce performance is driven by the interplay between payment behaviour, logistics efficiency and customer purchasing patterns. This notebook focuses on the **finance** angle: monthly revenue health, YoY/MoM trends, and the category / geography / payment-type breakdowns that support finance and commercial decisions.

## Strategy
1. **Revenue definition** - `price + freight_value` aggregated from `raw_order_items`. Payments are reconciled against this total but not used as the primary revenue signal (vouchers, partial payments can distort).
2. **Status filter** - orders with status `canceled` or `unavailable` are flagged via `is_revenue_recognised = 0` in the Silver fact and filtered out in every Gold mart.
3. **Ghost-order defence** - `orders_revenue_silver` uses an **inner join** on items; any payment pointing to an order that does not exist in `raw_orders` is dropped before Gold.
4. **1-to-many payments** - multiple payment rows are rolled up in `payments_order_agg_silver`; `primary_payment_type` is the type with the largest single payment.
5. **Rolling revenue / YoY / MoM** - month-based self-joins on `add_months(..., 1)` / `add_months(..., 12)` compute MoM and YoY (robust to sparse months); `Window.rowsBetween` computes rolling 3m / 6m revenue in `gold_revenue_monthly`.

## Business questions answered below
1. What is the monthly revenue trend (total, MoM %, YoY %, rolling 3m)?
2. Which product categories drive revenue, and how has the mix evolved?
3. Which customer states drive revenue, and how concentrated is it?
4. What is the primary-payment-type mix and how does AOV vary by method?
5. What does the single-row executive summary say?


## 1. Sanity checks on Bronze

Confirm the four tables UC1 depends on are present and populated.

In [0]:
%sql
-- EXPECTED: every count > 0 (bronze ingestion populated all five source tables).
SELECT 'raw_orders'          AS tbl, COUNT(*) AS rows FROM data_sentinals.bronze.raw_orders          UNION ALL
SELECT 'raw_order_items'     , COUNT(*)               FROM data_sentinals.bronze.raw_order_items     UNION ALL
SELECT 'raw_order_payments'  , COUNT(*)               FROM data_sentinals.bronze.raw_order_payments  UNION ALL
SELECT 'raw_products'        , COUNT(*)               FROM data_sentinals.bronze.raw_products        UNION ALL
SELECT 'raw_product_names_translation', COUNT(*)      FROM data_sentinals.bronze.raw_product_names_translation

### 1a. Negative-payment check (spec requirement)

Count how many rows we expect the silver layer to drop. Running this **before** silver is built shows the raw quality of the source.

In [0]:
%sql
-- EXPECTED: negative_payments > 0 in raw Bronze (proves we need the silver DQ rule).
-- These rows get dropped by @dlt.expect_or_drop in silver/revenue/revenue_payments.py.
SELECT
  SUM(CASE WHEN payment_value < 0 THEN 1 ELSE 0 END) AS negative_payments,
  SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS null_payments,
  MIN(payment_value) AS min_value,
  MAX(payment_value) AS max_value
FROM data_sentinals.bronze.raw_order_payments

### 1b. Ghost-order check on payments

Payments whose `order_id` does not exist in `raw_orders` are "ghost" payments; inner join in silver should eliminate them.

In [0]:
%sql
-- EXPECTED: ghost_payment_rows may be > 0 in Bronze (these are payments whose
-- order_id has no matching line item). The INNER JOIN in silver/revenue/revenue_orders.py
-- eliminates them before Gold - see assertion 2 in the smoke-test cell at the end.
SELECT COUNT(*) AS ghost_payment_rows
FROM data_sentinals.bronze.raw_order_payments p
LEFT ANTI JOIN data_sentinals.bronze.raw_orders o
  ON p.order_id = o.order_id

## 2. Silver validations

After the DLT pipeline has run the silver tables, run these spot checks.

In [0]:
%sql
-- 2.1  No negative payments survived into silver
-- EXPECTED: 0.  Proof the silver DQ rule actually fires.
SELECT COUNT(*) AS negative_payments_in_silver
FROM data_sentinals.silver.payments_clean_silver
WHERE payment_value < 0

In [0]:
%sql
-- 2.2  Order-grain uniqueness for the payments rollup
-- EXPECTED: rows == unique_orders (exactly one row per order_id after the
-- 1-to-many collapse). Any difference means the aggregation broke.
SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS unique_orders
FROM data_sentinals.silver.payments_order_agg_silver

In [0]:
%sql
-- 2.3  Payment-mix distribution - sanity check on primary_payment_type
SELECT primary_payment_type, COUNT(*) AS orders, ROUND(SUM(total_payment_value), 2) AS total_value
FROM data_sentinals.silver.payments_order_agg_silver
GROUP BY primary_payment_type
ORDER BY orders DESC

In [0]:
%sql
-- 2.4  Revenue fact - row count, coverage, reconciliation
-- EXPECTED: recognised_orders ~= 0.97 * orders (the rest are canceled/unavailable),
--           pct_reconciled_within_5pct >= 95 (most orders have payments tying out
--           to the item revenue within 5%; vouchers/partials explain the tail).
SELECT
  COUNT(*)                                                AS orders,
  SUM(is_revenue_recognised)                              AS recognised_orders,
  ROUND(SUM(revenue_total), 2)                            AS revenue_total,
  ROUND(SUM(total_payment_value), 2)                      AS payments_total,
  ROUND(AVG(ABS(payment_reconciliation_delta)), 2)        AS avg_abs_recon_delta,
  ROUND(AVG(CASE WHEN ABS(payment_reconciliation_delta) / GREATEST(revenue_total, 1) < 0.05 THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_reconciled_within_5pct
FROM data_sentinals.silver.orders_revenue_silver

In [0]:
%sql
-- 2.5  Category coverage - should be very little 'unknown'
-- EXPECTED: unknown_pct < 5.  Evidence that the Portuguese->English translation
-- coalesce in silver/revenue/revenue_product_catalog.py covers the vast majority of products.
SELECT
  SUM(CASE WHEN product_category_en = 'unknown' THEN 1 ELSE 0 END) AS unknown_items,
  COUNT(*) AS total_items,
  ROUND(SUM(CASE WHEN product_category_en = 'unknown' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS unknown_pct
FROM data_sentinals.silver.order_items_revenue_silver

## 3. Business Question 1 - Monthly revenue trend (MoM, YoY, rolling 3m)

This is the hero chart of the use case. The Gold mart has all the window calculations pre-computed.

In [0]:
%sql
SELECT
  order_purchase_month,
  total_orders,
  ROUND(total_revenue, 2)          AS total_revenue,
  ROUND(aov, 2)                    AS aov,
  ROUND(revenue_mom_pct, 2)        AS mom_pct,
  ROUND(revenue_yoy_pct, 2)        AS yoy_pct,
  ROUND(revenue_rolling_3m, 2)     AS rolling_3m,
  ROUND(revenue_rolling_6m, 2)     AS rolling_6m
FROM data_sentinals.gold.gold_revenue_monthly
ORDER BY order_purchase_month

### 3a. Where are the growth and shrink months?

In [0]:
%sql
SELECT
  order_purchase_month,
  ROUND(total_revenue, 2) AS total_revenue,
  ROUND(revenue_mom_pct, 2) AS mom_pct
FROM data_sentinals.gold.gold_revenue_monthly
WHERE revenue_mom_pct IS NOT NULL
ORDER BY revenue_mom_pct DESC
LIMIT 10

## 4. Business Question 2 - Revenue by product category

Which categories drive the business, and is the mix shifting over time?

In [0]:
%sql
-- Top 10 categories all-time
SELECT product_category_en,
       ROUND(SUM(total_revenue), 2) AS total_revenue,
       SUM(total_orders)            AS total_orders
FROM data_sentinals.gold.gold_revenue_by_category_month
GROUP BY product_category_en
ORDER BY total_revenue DESC
LIMIT 10

In [0]:
%sql
-- Category-rank movement across months (spot trend shifts)
SELECT order_purchase_month, product_category_en, category_rank_in_month,
       ROUND(revenue_share_of_month * 100, 2) AS share_pct
FROM data_sentinals.gold.gold_revenue_by_category_month
WHERE category_rank_in_month <= 5
ORDER BY order_purchase_month, category_rank_in_month

## 5. Business Question 3 - Revenue by customer state

Where is revenue concentrated? Useful for regional marketing and logistics planning.

In [0]:
%sql
SELECT customer_state,
       ROUND(SUM(total_revenue), 2) AS total_revenue,
       SUM(total_orders)            AS total_orders,
       ROUND(AVG(aov), 2)           AS avg_aov
FROM data_sentinals.gold.gold_revenue_by_state_month
GROUP BY customer_state
ORDER BY total_revenue DESC

## 6. Business Question 4 - Payment-type mix & AOV

Showcases the 1-to-many payments handling done in silver.

In [0]:
%sql
SELECT primary_payment_type,
       ROUND(SUM(total_revenue), 2)          AS total_revenue,
       SUM(total_orders)                     AS total_orders,
       ROUND(AVG(avg_installments), 2)       AS avg_installments,
       ROUND(AVG(share_of_month_revenue) * 100, 2) AS avg_share_of_month_pct
FROM data_sentinals.gold.gold_revenue_by_payment_type_month
GROUP BY primary_payment_type
ORDER BY total_revenue DESC

## 7. Business Question 5 - Executive summary

Single-row KPI snapshot for the slide deck.

In [0]:
%sql
SELECT * FROM data_sentinals.gold.gold_revenue_exec_summary

## 8. Dashboard guidance

Recommended Databricks dashboard tiles (all backed by the Gold marts above):

| Tile                | Source Gold table                       | Viz       |
|---------------------|------------------------------------------|-----------|
| Monthly revenue     | `gold_revenue_monthly`                   | line      |
| MoM % bars          | `gold_revenue_monthly`                   | bar       |
| Rolling 3m revenue  | `gold_revenue_monthly`                   | line      |
| Top categories      | `gold_revenue_by_category_month`         | treemap   |
| Revenue by state    | `gold_revenue_by_state_month`            | bar/map   |
| Payment mix         | `gold_revenue_by_payment_type_month`     | pie       |
| KPI scorecard       | `gold_revenue_exec_summary`              | counters  |

All tiles share the single publish schema `data_sentinals.gold`, which keeps the dashboard refresh simple.

## 9. UC1 smoke test (runnable assertions)

One-cell green/red check over the Gold + Silver state for Use Case 1.

Run this as the **last thing before the demo** - if every line prints `PASS`, the
pipeline is healthy and the dashboard is safe to open. If anything prints `FAIL`,
stop and investigate before presenting; the failing assertion name tells you
exactly which invariant broke.

The assertions codify the three spec requirements ("no negative payments", "no
ghost orders", "1-to-many payments collapse cleanly") plus four shape checks on
the Gold marts (share columns sum to 1, exec summary is a single row, category
coverage is high).

In [ ]:
CATALOG = "data_sentinals"

invariants = [
    (
        "silver_no_negative_payments",
        f"SELECT COUNT(*) FROM {CATALOG}.silver.payments_clean_silver WHERE payment_value < 0",
        0,
        "==",
    ),
    (
        "silver_payments_rollup_is_order_grain",
        f"""SELECT COUNT(*) - COUNT(DISTINCT order_id)
              FROM {CATALOG}.silver.payments_order_agg_silver""",
        0,
        "==",
    ),
    (
        "silver_no_ghost_orders_in_fact",
        f"""SELECT COUNT(*)
              FROM {CATALOG}.silver.orders_revenue_silver f
              LEFT ANTI JOIN {CATALOG}.bronze.raw_orders o
                ON f.order_id = o.order_id""",
        0,
        "==",
    ),
    (
        "silver_category_unknown_rate_under_5pct",
        f"""SELECT ROUND(100.0 * SUM(CASE WHEN product_category_en = 'unknown' THEN 1 ELSE 0 END) / COUNT(*), 2)
              FROM {CATALOG}.silver.order_items_revenue_silver""",
        5.0,
        "<",
    ),
    (
        "gold_category_share_sums_to_one_per_month",
        f"""SELECT COUNT(*) FROM (
              SELECT order_purchase_month,
                     ROUND(SUM(revenue_share_of_month), 2) AS s
              FROM {CATALOG}.gold.gold_revenue_by_category_month
              GROUP BY 1
            ) WHERE s <> 1.00""",
        0,
        "==",
    ),
    (
        "gold_state_share_sums_to_one_per_month",
        f"""SELECT COUNT(*) FROM (
              SELECT order_purchase_month,
                     ROUND(SUM(revenue_share_of_month), 2) AS s
              FROM {CATALOG}.gold.gold_revenue_by_state_month
              GROUP BY 1
            ) WHERE s <> 1.00""",
        0,
        "==",
    ),
    (
        "gold_payment_type_share_sums_to_one_per_month",
        f"""SELECT COUNT(*) FROM (
              SELECT order_purchase_month,
                     ROUND(SUM(share_of_month_revenue), 2) AS s
              FROM {CATALOG}.gold.gold_revenue_by_payment_type_month
              GROUP BY 1
            ) WHERE s <> 1.00""",
        0,
        "==",
    ),
    (
        "gold_exec_summary_is_single_row",
        f"SELECT COUNT(*) FROM {CATALOG}.gold.gold_revenue_exec_summary",
        1,
        "==",
    ),
]

print(f"Running {len(invariants)} UC1 invariants against catalog '{CATALOG}'\n")
passed, failed = 0, 0
for name, query, expected, op in invariants:
    got = spark.sql(query).first()[0]
    ok = (got == expected) if op == "==" else (got < expected) if op == "<" else False
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}]  {name:<48s}  expected {op} {expected}, got {got}")
    passed += int(ok)
    failed += int(not ok)

print(f"\nSummary: {passed} passed, {failed} failed")
assert failed == 0, f"{failed} UC1 invariant(s) failed - do NOT demo until these are green"